<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/TemperatureTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

자신의 모델을 선택한다.

In [1]:
LLM_PROVIDER = "openai" # claude 또는 gemini
KEYFROM_USERDATA = True  # 외부에서 입력하려면 False 로

필요한 모듈을 모두 설치한다.

In [ ]:
!pip install -qU langchain-openai langchain-anthropic langchain-google-genai

필요한 모듈을 설치한다.

In [3]:
# Colab용: Temperature 변화 실습 + 유사도 측정
import pandas as pd
from langchain_core.messages import SystemMessage, HumanMessage
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

외부에서 키를 입력받는다.

In [4]:
# API 키 입력
if KEYFROM_USERDATA:
  from google.colab import userdata
  API_KEY = userdata.get('OPENAI').strip()
else:
  import getpass
  API_KEY = getpass.getpass('Enter API key: ')

프롬프트를 설정한다.

In [5]:
# 공통 프롬프트
prompt = "비 오는 날 골목 카페에서 벌어지는 짧은 이야기를 3문장으로 써라."

messages = [
    SystemMessage(content="너는 짧은 문학적 글을 쓰는 작가다."),
    HumanMessage(content=prompt),
]

3가지 다른 temperature에 대해 결과를 비교해 본다.

In [6]:
# 비교할 temperature 값을 설정
temperatures = [0, 0.7, 1.5]

temperature별 결과를 저정할 딕셔너리를 준비한다.

In [7]:
all_outputs = {}

5번 호출하면서 결과가 어떻게 바뀌는지 관찰한다.<BR>
ChatGPT는 4o 버전 이후로 Tempertture를 지원하지 않는다. <BR>
따라서 여기서는 gpt-5.1 대신 4.1로 테스트 해 본다.

In [ ]:
for temp in temperatures:
    print("\n" + "=" * 60)
    print(f"Temperature = {temp}")
    print("=" * 60)

    if LLM_PROVIDER == "openai":
      from langchain_openai import ChatOpenAI

      llm = ChatOpenAI(
          model="gpt-4.1-nano",
          temperature=temp,
          api_key=API_KEY,
      )


    elif LLM_PROVIDER == "claude":
      from langchain_anthropic import ChatAnthropic

      llm = ChatAnthropic(
          model="claude-sonnet-4-6",
          temperature=temp,
          api_key=API_KEY,
      )


    elif LLM_PROVIDER == "gemini":
      from langchain_google_genai import ChatGoogleGenerativeAI

      llm = ChatGoogleGenerativeAI(
          model="gemini-2.5-flash",
          temperature=temp,
          api_key=API_KEY,
      )


    outputs = []

    for i in range(5):
        response = llm.invoke(messages)
        outputs.append(response.content)

        print(f"\n--- 실행 {i+1} ---")
        print(response.content)

    all_outputs[temp] = outputs

각 생성된 문장의 코사인 유사도를 계산해 본다.

In [ ]:
# 코사인 유사도 평균 계산
summary = []

for temp, outputs in all_outputs.items():
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(outputs)

    sim_matrix = cosine_similarity(X)

    # 자기 자신과의 유사도 1.0은 제외하고 평균 계산
    scores = []

    for i in range(5):
        for j in range(i + 1, 5):
            scores.append(sim_matrix[i][j])

    avg_similarity = sum(scores) / len(scores)

    summary.append({
        "temperature": temp,
        "average_cosine_similarity": round(avg_similarity, 3)
    })

    print("\n" + "=" * 60)
    print(f"Temperature = {temp} 코사인 유사도 행렬")
    print("=" * 60)

    df = pd.DataFrame(
        sim_matrix,
        index=[f"실행{i+1}" for i in range(5)],
        columns=[f"실행{i+1}" for i in range(5)]
    )

    display(df)

# 최종 요약표
summary_df = pd.DataFrame(summary)

최종 결과를 출력해 본다.

In [ ]:
print("\n" + "=" * 60)
print("Temperature별 평균 코사인 유사도")
print("=" * 60)

display(summary_df)